# FREYA: fine-tune her fallback voice on Google Colab

1. **Runtime > Change runtime type > T4 GPU**
2. Optional, to upload the result to your Hugging Face account: **Secrets** (key icon on the left) → add `HF_TOKEN`, a *write* token from huggingface.co/settings/tokens.
3. Choose the settings below and run the cell. It asks once for Google Drive, where checkpoints and the result are saved.

**What it does.** LoRA (QLoRA: the base is loaded in 4-bit) on an uncensored Qwen3.5 4B or 9B, trained only on FREYA's replies. It **tests her here** with the persona evaluation, saves `freya-trained.gguf` and the adapter to Drive, and can upload both to Hugging Face. 15-40 minutes on a T4. If the session is cut off, run it again: it resumes from the last checkpoint on Drive.

**What it changes, and what it doesn't.** Fine-tuning changes how she sounds (rhythm, habits, fewer assistant phrases), not what she knows or who she is. Her persona, memory and context come from the app on every message, whichever model answers. That's why this is her *fallback* voice: the stock 27B on the server notebook usually reasons better; keep a fine-tune only if it wins the blind test (`npm run bakeoff`).

**Your own conversations**: tick the box, then pick `.jsonl` files (format: `training/conversation-template.jsonl`). Every file is validated first: adults only, nothing sexual involving minors, no non-consent, nothing that teaches FREYA to claim she's human when sincerely asked. A problem stops the run with the line numbers.

| Setting | T4 (15 GB) | Notes |
|---|---|---|
| Base | 4B: fast, fits easily · 9B: better, fits at 4-bit | a 27B needs 24 GB+ to train |
| Max sequence | 2048 | longer conversations need more memory |
| Rank | 16-32 | higher learns more style, and memorises sooner |
| Epochs | 1-2 | more makes her repeat her examples |


In [ ]:
#@title Train, test and save FREYA
REPO_URL = "https://github.com/KrishAnjara/freya-v1.git"  #@param {type:"string"}
BASE_MODEL = "huihui-ai/Huihui-Qwen3.5-4B-abliterated"  #@param ["huihui-ai/Huihui-Qwen3.5-4B-abliterated", "huihui-ai/Huihui-Qwen3.5-9B-abliterated", "huihui-ai/Huihui-Qwen3-4B-Instruct-2507-abliterated"]
EPOCHS = 2  #@param {type:"number"}
LEARNING_RATE = 1e-4  #@param {type:"number"}
RANK = 32  #@param {type:"integer"}
MAX_SEQ = 2048  #@param {type:"integer"}
#@markdown Upload to your Hugging Face account (USER/NAME; empty to skip). Needs HF_TOKEN in Secrets.
HF_REPO = ""  #@param {type:"string"}
HF_PRIVATE = True  #@param {type:"boolean"}
#@markdown Also save a merged 16-bit model (large; only for vLLM or another quantisation):
MERGED = False  #@param {type:"boolean"}
#@markdown Quick check that everything loads (10 steps, nothing saved):
SMOKE_TEST = False  #@param {type:"boolean"}
UPLOAD_OWN_DATA = False  #@param {type:"boolean"}

import os
from google.colab import drive
drive.mount("/content/drive")
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
except Exception:
    pass  # No secret: everything works except the upload.
if os.path.isdir("/content/freya-v1/.git"):
    !git -C /content/freya-v1 pull -q
else:
    !git clone -q {REPO_URL} /content/freya-v1
%cd /content/freya-v1
if UPLOAD_OWN_DATA:
    from google.colab import files
    os.makedirs("training/data", exist_ok=True)
    for name, content in files.upload().items():
        with open(f"training/data/{os.path.basename(name)}", "wb") as handle:
            handle.write(content)
flags = " ".join(filter(None, [
    "--smoke" if SMOKE_TEST else "",
    f"--hf-repo {HF_REPO}" if HF_REPO else "",
    "--hf-private" if HF_REPO and HF_PRIVATE else "",
    "--merged" if MERGED else "",
]))
!python training/colab_run.py --base "{BASE_MODEL}" --epochs {EPOCHS} --learning-rate {LEARNING_RATE} --rank {RANK} --max-seq {MAX_SEQ} --save-to /content/drive/MyDrive/freya {flags}

## Then: use it, without downloading anything to your laptop

- **As her fallback on the server:** in `freya_server.ipynb`, set `FALLBACK_MODEL` to `hf.co/USER/NAME` (if you uploaded it as a public repo), or run the server with `--fallback-model /content/drive/MyDrive/freya/freya-trained.gguf`.
- **As her main voice** (to try it): set `MODEL` to `hf:USER/NAME/freya-trained.gguf` (private repos work with `HF_TOKEN` in Secrets), or to the Drive path above.
- Then `npm run bakeoff` on your laptop compares it blind against the stock model. Keep it only if it wins.

If you do want it on the laptop after all: put `freya-trained.gguf` in `training/freya-gguf/` and run `start.bat`; it builds the local model from that file. To go back to the stock model, delete the file and run `npm run ollama:create`.